# 00. Setup and Import Existing Models
Load the required libraries and set up the environment.
Additionally, download Glove vectors if not already present.


In [46]:
# Importing necessary libraries
import os
import sys
import pickle
from urllib.request import urlopen, urlretrieve
import zipfile
import deepl
import numpy as np

# Read environment variables
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")

# Clients
deepl_client = deepl.DeepLClient(DEEPL_API_KEY)

# External links
url_glove = "https://nlp.stanford.edu/data/glove.6B.zip"
urls_chisco = open("chisco_pkl_urls.txt").read().splitlines()

In [64]:
def get_glove_embeddings(glove_url):
    # Download from https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip using os process
    urlretrieve(glove_url, "glove.zip")
    with zipfile.ZipFile("glove.zip", "r") as zip_ref:
        zip_ref.extractall(".")

    path_to_glove_file = "glove.6B.100d.txt"
    embeddings_index = {}
    with __builtins__.open(path_to_glove_file) as f:
        for line in f:
            word, coefs = line.split(maxsplit=1)
            coefs = np.fromstring(coefs, "f", sep=" ")
            embeddings_index[word] = coefs
    return embeddings_index

def get_sentence_embeddings(text):
    # For each word in the text, get the embedding
    words = text.split()
    embeddings = []
    for word in words:
        if word in glove_embeddings:
            embeddings.append(glove_embeddings[word])
        else:
            embeddings.append(np.zeros(100))  # Assuming 100-dimensional embeddings
    return np.mean(embeddings, axis=0)

glove_embeddings = get_glove_embeddings(url_glove)

# 01. Data Download and Preprocessing
Data is obtained from the [Chisco Dataset](https://www.nature.com/articles/s41597-024-04114-1). For performance reasons, only the .pkl files are downloaded.

In [65]:
def preprocess_picke_from_url(url):
    """
    Function to parse pickle file from a given URL.
    """
    try:
        with urlopen(url) as response:
            obj = pickle.load(response)
            for p in obj:
                p["translated_text"] = deepl_client.translate_text(p["text"], 
                                                                   target_lang="EN-US",
                                                                   context="Translate into as few words as possible.").text
                p["eeg_features"] = p["input_features"]
                p["embeddings"] = get_sentence_embeddings(p["translated_text"])
                p["input_features"] = []
                del p["input_features"]
            return obj
    except Exception as e:
        print(f"Error loading pickle file from {url}: {e}")
        return None

In [66]:
pickles = [
    preprocess_picke_from_url(url) for url in urls_chisco[0:1]
]

In [67]:
pickles[0][34]

{'text': '不敢照看孩子',
 'translated_text': 'Afraid to babysit',
 'eeg_features': array([[[ 8.46311476e-06,  9.58281242e-06,  7.70149001e-06, ...,
          -5.65671055e-07, -3.13355380e-06,  3.97269035e-07],
         [ 4.99860432e-06,  6.27518911e-06,  4.16211508e-06, ...,
          -1.24706871e-06, -1.51274747e-06,  1.13971038e-06],
         [ 4.89646685e-06,  6.81745397e-06,  4.10159837e-06, ...,
          -6.04104435e-08, -4.26015323e-07,  1.52530187e-06],
         ...,
         [ 5.78995829e+00,  4.51047263e+00,  4.69619110e+00, ...,
           5.52190231e+00,  4.61104674e+00,  4.92903258e+00],
         [ 3.01375488e+00, -2.70613732e-01,  7.65593251e-01, ...,
           3.35462754e+00,  3.29092531e+00,  1.01376052e+00],
         [ 6.52800000e+04,  6.52800000e+04,  6.52800000e+04, ...,
           6.52800000e+04,  6.52800000e+04,  6.52800000e+04]]],
       shape=(1, 125, 1651)),
 'embeddings': array([-0.11642667,  0.03032367, -0.17350666,  0.06156867, -0.064429  ,
         0.33452334, -0